In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Set File Paths (Already uploaded in Colab)
# ---------------------------------------------------------
spoofed_path = "spoofed_traj.txt"
gps_path = "gps_positions.csv"

# ---------------------------------------------------------
# 2. Load data
# ---------------------------------------------------------
spoofed_cols = ["time", "x", "y", "z", "q_x", "q_y", "q_z", "q_w"]

# Handle both space-delimited .txt and standard .csv seamlessly
spoofed = pd.read_csv(spoofed_path, sep=r'\s+', header=None, names=spoofed_cols)

gps = pd.read_csv(gps_path)
gps.columns = [c.strip() for c in gps.columns]  # normalize headers
gps = gps.sort_values("Time_s").reset_index(drop=True)

# ---------------------------------------------------------
# 3. Apply transformation matrix T to remap spoofed axes
#    into GPS convention: [x_m, y_m, z_m]^T = T @ [x, y, z]^T
# ---------------------------------------------------------
T = np.array([
    [0, -1,  0],
    [0,  0, -1],
    [1,  0,  0],
])

xyz = spoofed[["x", "y", "z"]].to_numpy()          # shape (N, 3)
xyz_transformed = (T @ xyz.T).T                     # shape (N, 3)

spoofed["x_m"] = xyz_transformed[:, 0]
spoofed["y_m"] = xyz_transformed[:, 1]
spoofed["z_m"] = xyz_transformed[:, 2]

# ---------------------------------------------------------
# 4. AsOf/Left-merge: attach spoofed data to GPS timestamps
# ---------------------------------------------------------
ROUND_DECIMALS = 6
spoofed["time_r"] = spoofed["time"].round(ROUND_DECIMALS)
gps["time_r"] = gps["Time_s"].round(ROUND_DECIMALS)

full = pd.merge(gps, spoofed, on="time_r", how="left")
full = full.sort_values("Time_s").reset_index(drop=True)

if full["x_m"].notna().sum() == 0:
    raise ValueError(
        "No matching timestamps between spoofed trajectory and gps_positions.csv. "
        "Check that the time columns actually align."
    )

# ---------------------------------------------------------
# 5. Compute raw 2D error magnitude/direction using
#    TRANSFORMED spoofed coordinates (x_m, y_m) vs GPS (GPS_X, GPS_Y)
# ---------------------------------------------------------
full["err_dx"] = full["x_m"] - full["GPS_X"]
full["err_dy"] = full["y_m"] - full["GPS_Y"]
full["err_mag"] = np.sqrt(full["err_dx"]**2 + full["err_dy"]**2)
full["err_dir_deg"] = np.degrees(np.arctan2(full["err_dy"], full["err_dx"]))

# ---------------------------------------------------------
# Helper Functions
# ---------------------------------------------------------
def circular_mean_deg(angles_deg):
    angles_rad = np.radians(angles_deg)
    mean_sin = np.mean(np.sin(angles_rad))
    mean_cos = np.mean(np.cos(angles_rad))
    return np.degrees(np.arctan2(mean_sin, mean_cos))

def circular_diff_deg(a, b):
    """Smallest signed difference a-b, wrapped to (-180, 180]."""
    return (a - b + 180) % 360 - 180

def filtered_window_stats(mags, dirs, mag_thresh=1.0, dir_thresh_deg=45.0):
    mags = np.asarray(mags, dtype=float)
    dirs = np.asarray(dirs, dtype=float)

    mean_mag = np.mean(mags)
    mean_dir = circular_mean_deg(dirs)

    mag_ok = np.abs(mags - mean_mag) <= mag_thresh
    dir_ok = np.abs(circular_diff_deg(dirs, mean_dir)) <= dir_thresh_deg
    keep = mag_ok & dir_ok

    if not np.any(keep):
        return mean_mag, mean_dir  # fallback: unfiltered window mean

    return np.mean(mags[keep]), circular_mean_deg(dirs[keep])

# ---------------------------------------------------------
# 6. Build corrected trajectory for ALL GPS rows
# ---------------------------------------------------------
def build_new_traj(full, n, a):
    n = int(n)
    N = len(full)

    new_x = full["GPS_X"].copy().astype(float)
    new_y = full["GPS_Y"].copy().astype(float)

    mags_all = full["err_mag"].values
    dirs_all = full["err_dir_deg"].values

    last_valid_stats = None

    for t in range(n, N):
        window_mags = mags_all[t - n : t]
        window_dirs = dirs_all[t - n : t]

        valid_mask = ~np.isnan(window_mags) & ~np.isnan(window_dirs)

        if np.any(valid_mask):
            filt_mag, filt_dir = filtered_window_stats(
                window_mags[valid_mask], window_dirs[valid_mask]
            )
            last_valid_stats = (filt_mag, filt_dir)
        elif last_valid_stats is not None:
            filt_mag, filt_dir = last_valid_stats
        else:
            continue

        applied_mag = filt_mag + a
        theta = np.radians(filt_dir)

        dx = applied_mag * np.cos(theta)
        dy = applied_mag * np.sin(theta)

        new_x.iloc[t] = full["GPS_X"].iloc[t] + dx
        new_y.iloc[t] = full["GPS_Y"].iloc[t] + dy

    new_traj = pd.DataFrame({
        "Time_s": full["Time_s"],
        "GPS_X": new_x,
        "GPS_Y": new_y,
        "GPS_Z": full["GPS_Z"],
    })
    return new_traj

# ---------------------------------------------------------
# 7. User-defined parameters
# ---------------------------------------------------------
n = int(input("Enter window size n: "))
a = float(input("Enter offset a (meters): "))

new_traj = build_new_traj(full, n, a)

# ---------------------------------------------------------
# 8. Save and download
# ---------------------------------------------------------
out_path = "new_traj.csv"
new_traj.to_csv(out_path, index=False)
print(f"Saved {out_path} with {len(new_traj)} rows (original GPS row count preserved).")
files.download(out_path)


Enter window size n: 5
Enter offset a (meters): 0.2
Saved new_traj.csv with 4543 rows (original GPS row count preserved).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>